# Data processing for Big data

##  Activity: Parallel Search

For this tutorial, we will focus on parallel search in Big Data. Thus, the following sections will be done:
1. Review Data partitioning strategies
2. Implement distinct searching functionalities using RDDs: 
3. Implement distinct searching functionalities using Spark SQL module: you will use the Spark API to use dataframes and Spark SQL to perform the search functionality as in section 1.


Also, you will need to  visualise the parallelism on searching in these APIs and RDD implementation. Furthermore, you will need to look at the Query execution plan done by the Spark Optimizer Engine and understand how internally Spark executes or plans a searching function.

Let's get started.

## Table of Contents

* [SparkContext and SparkSession](#one)
* [Data Partitioning](#two)
* [Spark RDDs](#three)
    * [Data Partitioning in RDD](#three)
        * [Default Partitioning](#default)
        * [Hash Partitioning](#hash)
        * [Range Partitioning](#range)
    * [Parallel Search in RDDs](#parallel-search-rdd)    
* [Spark DataFrames](#dataframes)
    * [Data Partitioning in DataFrames](#df-partitioning)
    * [Parallel Search in DataFrames](#parallel_search_df)    
    * [Parallel Search with SparkSQL](#parallel_search_sparksql)       
* [Lab Tasks](#lab-task-1)
    * [Lab Task 1](#lab-task-1)
    * [Lab Task 2](#lab-task-2)
    * [Lab Task 3](#lab-task-3)
    * [Lab Task 4](#lab-task-4)


## SparkContext and SparkSession <a class="anchor" name="one"></a>

In [1]:
# Import SparkConf class into program
from pyspark import SparkConf

# local[*]: run Spark in local mode with as many working processors as logical cores on your machine
# If we want Spark to run locally with 'k' worker threads, we can specify as "local[k]".
master = "local[*]"
# The `appName` field is a name to be shown on the Spark cluster UI page
app_name = "Parallel Search"
# Setup configuration parameters for Spark
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

# Import SparkContext and SparkSession classes
from pyspark import SparkContext # Spark
from pyspark.sql import SparkSession # Spark SQL

# Method 1: Using SparkSession
spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel('ERROR')

# # Method 2: Getting or instantiating a SparkContext
# sc = SparkContext.getOrCreate(spark_conf)
# sc.setLogLevel('ERROR')

In [3]:
data = spark.sparkContext.parallelize(range(10))
print(data.collect())

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 2 in stage 0.0 failed 1 times, most recent failure: Lost task 2.0 in stage 0.0 (TID 2) (JiayiLiu07 executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(Interr

In [2]:
spark.version

'3.5.5'

## Data Partitioning <a class="anchor" id="two"></a>
In this first part of the tutorial, we will do a quick review of a few data partitioning strategies which we will need to know for the rest of the topics in this tutorial.

Data partitioning is the fundamental step for parallel search algorithms as parallelism in query and search processing is achieved through data partionining. 
In this activity, we will consider the following **three** partitioning strategies:
#### 1. Round-robin data partitioning ###
Round-robin data partitioning is the simplest data partitioning method in which each record in turn is allocated to a processing element (simply processor). Since it distributes the data evenly among all processors, it is also known as "equal-partitioning".

#### 2. Range data partitioning ###
Range data partitioning records based on a given range of the partitioning attribute. For example,the student table is partitioned based on "Last Name" based on the alphabetical order (i.e. A ~ Z). 

#### 3. Hash data partitioning ###
Hash data partitioning makes a partition based on a particular attribute using a hash function. The result of a hash function determines the processor where the record will be placed. Thus, all records within a partition have the same hash value.

## RDD partitioning <a class="anchor" id="three"></a>

By default, Spark partitions the data using <strong>Random equal partitioning</strong> unless there are specific transformations that uses a different type of partitioning</strong>
In the code below, we have defined two functions to implement custom partitioning using <strong>Range Partitioning</strong> and <strong>Hash Partitioning</strong>.


In [3]:
from pyspark.rdd import RDD

#A Function to print the data items in each RDD
#WARNING: this function is only for demo purpose, it should not be used on large dataset
def print_partitions(data):
    if isinstance(data, RDD):
        numPartitions = data.getNumPartitions()
        partitions = data.glom().collect()
    else:
        numPartitions = data.rdd.getNumPartitions()
        partitions = data.rdd.glom().collect()
    
    print(f"####### NUMBER OF PARTITIONS: {numPartitions}")
    for index, partition in enumerate(partitions):
        # show partition if it is not empty
        # if len(partition) > 0:
        #     print(f"Partition {index}: {len(partition)} records")
        #     print(partition)
        print(f"Partition {index}: {len(partition)} records")
        print(partition)

In [4]:
#Sample data used for demonstrating the partitioning
list_tutors = [(1,'Aaditya'),(2,'Chinnavit'),(3,'Neha'),(4,'Huashun'),(5,'Mohammad'),
                (10,'Peter'),(11,'Paras'),(12, 'Tooba'),(3, 'David'),(18,'Cheng'),(9,'Haqqani')]

#Define the number of partitions
no_of_partitions = 4

### Default Partitioning in Spark RDD <a class="anchor" id="default"></a>

In [7]:
# random equal partition
rdd = sc.parallelize(list_tutors, no_of_partitions)

In [8]:
rdd.glom().collect()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 3 in stage 0.0 failed 1 times, most recent failure: Lost task 3.0 in stage 0.0 (TID 3) (192.168.110.238 executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(I

In [7]:
print("Number of partitions:{}".format(rdd.getNumPartitions()))
print("Partitioner:{}".format(rdd.partitioner))


Number of partitions:4
Partitioner:None


In [11]:
!pip show py4j

Name: py4j
Version: 0.10.9.7
Summary: Enables Python programs to dynamically access arbitrary Java objects
Home-page: https://www.py4j.org/
Author: Barthelemy Dagenais
Author-email: barthelemy@infobart.com
License: BSD License
Location: C:\Users\86137\AppData\Roaming\Python\Python312\site-packages
Requires: 
Required-by: pyspark


In [10]:
rdd.glom().collect()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 3 in stage 2.0 failed 1 times, most recent failure: Lost task 3.0 in stage 2.0 (TID 11) (192.168.110.238 executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#006DAE">TODO: </strong>How do you think the data is divided across the partitions by default when no partitoner is specified?</div>

In [7]:
"""
Answer:
default partition method is Hash Partition
"""

'\nAnswer:\ndefault partition method is Hash Partition\n'

### Hash Partitioning in RDD <a class="anchor" id="hash"></a>
Hash partitioning uses the formula <code>partition = hash_function() % numPartitions</code> to determine which partition data item falls into.

In [8]:
#Hash Function to implement Hash Partitioning 
#Just computes the sum of digits
#Example : hash_function(12) produces 3 i.e. 2 + 1
def hash_function(key):
    total = 0
    for digit in str(key):
        total += int(digit)
    return total

In [9]:
# hash partitioning
hash_partitioned_rdd = rdd.partitionBy(no_of_partitions, hash_function)
print_partitions(hash_partitioned_rdd)            

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 2.0 failed 1 times, most recent failure: Lost task 1.0 in stage 2.0 (TID 9) (192.168.110.238 executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(I

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#006DAE">Note: </strong>Look at how the data is partitioned. For example, Partition 0 has 1 record, [(4, 'Huashun')]. Here is the step-wise breakdown:
    <ul>
        <li>hash_function(4) = 4</li>
        <li>Partition for the key of 4 is determined by <code>hash_function(4)%numPartitions</code> i.e. 4%4=0</li>
        <li>Similarly, for (18,'Cheng'), partition is given by <code>hash+function(18)%numPartitions</code> i.e. 9%4=1</li>
    </ul>

</div>

### Range Partitioning in RDD <a class="anchor" id="range"></a>
This strategy uses a range to distribute the items to respective partitions when the keys fall within the range. 

In [10]:
no_of_partitions=4

#Find the size of the elements in RDD
chunk_size = len(list_tutors)/no_of_partitions
#Define a range of values by key to distribute across partitions
#Here for simplicity, we are defining the range i.e. keys from 1-4 to fall in first partition, 5-9 in second partition and so on
range_arr=[[1,4],[5,9],[10,14],[15,19]]

def range_function(key):
    for index,item in enumerate(range_arr):
        if key >=item[0] and key <=item[1]:
            return index



In [12]:
# range partition
range_partitioned_rdd = rdd.partitionBy(no_of_partitions, range_function)
print_partitions(range_partitioned_rdd)

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 6.0 failed 1 times, most recent failure: Lost task 0.0 in stage 6.0 (TID 16) (192.168.110.238 executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:612)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:594)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:789)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:766)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.hasNext(PythonRunner.scala:525)
	at org.apache.spark.InterruptibleIterator.hasNext(

## Parallel Search using RDDs  <a class="anchor" id="parallel-search-rdd"></a>

Now we will implement basic search functionalities and visualise the parallelism embedded in Spark to perform these kind of queries.

In this tutorial, you will use a csv dataset **bank.csv**. However, for this tutorial we won't analyse the case study but only perform some search queries with this data

In [13]:
# Using Spark, we can read and load a csv file
# Read csv file and load into an RDD object
bank_rdd = sc.textFile('bank.csv')

# If you want to specify the number of partitions, you can add the number as a second argument
# bank_rdd = sc.textFile('bank.csv', 10)

## Exploring the data file, we can see that it contains different types of information
## Some useful information is printed below
print(f"Total partitions: {bank_rdd.getNumPartitions()}")
print(f"Number of lines: {bank_rdd.count()}")

## Each element of the RDD is a line from the file
bank_rdd.take(4)

Py4JJavaError: An error occurred while calling o137.partitions.
: java.lang.UnsatisfiedLinkError: org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Ljava/lang/String;I)Z
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:793)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1249)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1454)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:601)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:761)
	at org.apache.hadoop.fs.Globber.listStatus(Globber.java:128)
	at org.apache.hadoop.fs.Globber.doGlob(Globber.java:291)
	at org.apache.hadoop.fs.Globber.glob(Globber.java:202)
	at org.apache.hadoop.fs.FileSystem.globStatus(FileSystem.j

### Search in RDDs based on multiple conditions

We will focus on only four attributes from the data: age, education, marital and balance for filtering conditions. However, we will display additional information as well.

In [8]:
# 1. Split each line separated by comma into a list 
bank_rdd1 = bank_rdd.map(lambda line: line.split(','))
# 2. Remove the header
header = bank_rdd1.first()
bank_rdd1 = bank_rdd1.filter(lambda row: row != header)   #filter out header

# Indices for each attribute we will use
# Filter: age, education, marital, balance = 0, 3, 2, 5
# Display additional: day, month, deposit = 9, 10, 16

# 3. Search the records with balance between 1000 and 2000
bank_rdd1 = bank_rdd1.filter(lambda x: int(x[5])>1000 and int(x[5])<2000)
# 4. Also search the records with primrary or secondary education and age less than 30
bank_rdd1 = bank_rdd1.filter(lambda x: x[3] in ['primary','secondary'] and int(x[0])<30)
# 5. Also filter with those who are married
bank_rdd1 = bank_rdd1.filter(lambda x: x[2]=='married' )
# 6. Display the previous attributes plus the information of day, month and deposit
bank_rdd1 = bank_rdd1.map(lambda field: (field[0],field[2],field[3],field[5],
                                         field[9],field[10],field[16]))
# Print how many final records
print(bank_rdd1.count())

NameError: name 'bank_rdd' is not defined

In [9]:
# Let's see how the data was divided and the data for each partition
numPartitions = bank_rdd1.getNumPartitions()
print(f"Total partitions: {numPartitions}")

# glom(): Return an RDD created by coalescing all elements within each partition into a list
# WARNING: glom().collect() only works for a small dataset
partitions = bank_rdd1.glom().collect()
for index,partition in enumerate(partitions):
    print(f'------ Partition {index}:')
    for record in partition:
        print(record)

NameError: name 'bank_rdd1' is not defined

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#006DAE">TODO: </strong>Verify the parallelism in the Spark UI and explore the content. How many jobs have been executed so far?</div>

In [3]:
sc

<SparkContext master=local[*] appName=Parallel Search>

### Searching max/min value of an attribute in an RDD
This task will aim to find the record in the dataset that contains the highest value for a given attribute. In this case the attribute chosen is "balance".

In [12]:
# Read csv but now with 4 partitions
bank_rdd_4 = sc.textFile('bank.csv',4)

# Split and remove the header
bank_rdd_4 = bank_rdd_4.map(lambda line: line.split(','))
header = bank_rdd_4.first()
bank_rdd_4 = bank_rdd_4.filter(lambda row: row != header)   #filter out header

# Display the first 3 records
bank_rdd_4.take(3)

[['59',
  'admin.',
  'married',
  'secondary',
  'no',
  '2343',
  'yes',
  'no',
  'unknown',
  '5',
  'may',
  '1042',
  '1',
  '-1',
  '0',
  'unknown',
  'yes'],
 ['56',
  'admin.',
  'married',
  'secondary',
  'no',
  '45',
  'no',
  'no',
  'unknown',
  '5',
  'may',
  '1467',
  '1',
  '-1',
  '0',
  'unknown',
  'yes'],
 ['41',
  'technician',
  'married',
  'secondary',
  'no',
  '1270',
  'yes',
  'no',
  'unknown',
  '5',
  'may',
  '1389',
  '1',
  '-1',
  '0',
  'unknown',
  'yes']]

In [13]:
# Using the RDD function max(), it can be obtained in a single line 
result_max_balance = bank_rdd_4.max(key=lambda x: x[5]) # Get max by value in index 5 (balance)
# Print the record obtain with highest balance
print(result_max_balance)

['58', 'self-employed', 'married', 'secondary', 'no', '9994', 'no', 'no', 'cellular', '10', 'jul', '400', '1', '-1', '0', 'unknown', 'no']


In [19]:
# Get record with balance 10576
bank_record = bank_rdd_4.filter(lambda x: x[5]=='10576').collect()
print(bank_record)

[['29', 'management', 'married', 'tertiary', 'no', '10576', 'no', 'no', 'unknown', '15', 'may', '1224', '2', '-1', '0', 'unknown', 'yes']]


<a class="anchor" id="lab-task-1"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">1. Lab Task: </strong>Compare the <code>result_max_balance</code> record with the record above <code>(bank_record)</code>. Was the record obtained previously correct i.e. <code>result_max_balance</code>? <b>Explain what happened.</b></div>


In [20]:
result_max_balance = bank_rdd_4.max(key=lambda x: x[5])

In [21]:
result_max_balance[5]

'9994'

In [22]:
"""
发现：
这里的余额显然9994是小于10576的
但是我们取出来的余额最大值确是9994,显然逻辑存在问题

原因：
余额的数据类型给的是string 而不是 int 
string的比较方式，不是比整个数字的大小，转换成int可以解决问题
"""

'\n发现：\n这里的余额显然9994是小于10576的\n但是我们取出来的余额最大值确是9994,显然逻辑存在问题\n\n原因：\n余额的数据类型给的是string 而不是 int \nstring的比较方式，不是比整个数字的大小，转换成int可以解决问题\n'

<a class="anchor" id="lab-task-2"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">2. Lab Task: </strong>As you noticed in the previous result, the record returned originally (result_max_balance) was incorrect.
    <p><i>Fix the code below that uses the <code>max()</code> function to get the record with the correct maximum balance.</p></i></div>


In [14]:
# Using the RDD function max(), it can be obtained in a single line 
result_max_balance = bank_rdd_4.max(key=lambda x: int(x[5])) # Get max by value in index 5 (balance)
# Print the record obtain with highest balance
print(result_max_balance)

NameError: name 'bank_rdd_4' is not defined

#### To learn more about functions in RDDs, you can look into the next 2 sites:
1. http://spark.apache.org/docs/latest/rdd-programming-guide.html#rdd-operations
1. https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_Cheat_Sheet_Python.pdf

## DataFrames in Spark <a class="anchor" id="dataframes"></a>
A DataFrame is a distributed collection of data organized into named columns. It is equivalent to a table in relational database or a dataframe in R/Python but with richer optimizations under the hood. For more information visit : 

https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html

### Creating DataFrames
SparkSession provides an easy method <code>createDataFrame</code> to create Spark DataFrames. Data can be loaded from csv, json, xml and other sources like local file system or HDFS. More information on : 
https://spark.apache.org/docs/latest/api/python/pyspark.sql.html

To display the schema, i.e. the  structure of the DataFrame, you can use <strong>printSchema()</strong> method.

In [ ]:
df = spark.createDataFrame([(1,'Aaditya','A'),(2,'Chinnavit','C'),(3,'Neha','N'),(4,'Huashun','H'),(5,'Mohammad','M'),
                            (10,'Prajwol', 'P'),(1,'Paras','P'),(1, 'Tooba','T'),(3, 'David','D'),(4,'Cheng','C'),(9,'Haqqani','H')],
                           ['Id','Name','Initial'])

#display the rows of the dataframe
df.show(5)
# view the schema
# meta data -> data type of each column
df.printSchema()

+---+---------+-------+
| Id|     Name|Initial|
+---+---------+-------+
|  1|  Aaditya|      A|
|  2|Chinnavit|      C|
|  3|     Neha|      N|
|  4|  Huashun|      H|
|  5| Mohammad|      M|
+---+---------+-------+
only showing top 5 rows

root
 |-- Id: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Initial: string (nullable = true)



Another way to create a DataFrame is use the <strong>spark.read.csv</strong> file to load the data from csv to a DataFrame

In [ ]:
df = spark.read.csv("bank.csv",header=True)

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#006DAE">TODO: </strong>Display first 10 rows of the above dataframe. 
Try out other dataframe methods:    
    <ul>
        <li><code>df.columns, df.count()</code></li>
        <li><code>df.describe('column_name').show()</code></li>
        <li><strong>Selecting:</strong><code>df.select('column_name').show(), df.select('column_name').distinct().show()</code>
        </li>
        <li><strong>Filtering:</strong><code>df.filter(df.column_name == 123).show()</code></li>
</div>

In [ ]:
df.head(10) 
df.tail(10) 

### Partitioning in DataFrames <a class="anchor" id="df-partitioning"></a>

In [5]:
df = spark.createDataFrame([(1,'Aaditya','A'),(2,'Chinnavit','C'),(3,'Neha','N'),(4,'Huashun','H'),(5,'Mohammad','M'),
                            (10,'Prajwol', 'P'),(1,'Paras','P'),(1, 'Tooba','T'),(3, 'David','D'),(4,'Cheng','C'),(9,'Haqqani','H')],
                           ['Id','Name','Initial'])

In [10]:
df = spark.createDataFrame([
    (1, "Alice", 25),
    (2, "Bob", 30),
    (3, "Charlie", 35),
    (4, "Dave", 40),
    (5, "Eve", 45),
    (6, "Frank", 50)
], ["id", "name", "age"])

print_partitions(df)

####### NUMBER OF PARTITIONS: 14
Partition 0: 0 records
[]
Partition 1: 0 records
[]
Partition 2: 1 records
[Row(id=1, name='Alice', age=25)]
Partition 3: 0 records
[]
Partition 4: 1 records
[Row(id=2, name='Bob', age=30)]
Partition 5: 0 records
[]
Partition 6: 1 records
[Row(id=3, name='Charlie', age=35)]
Partition 7: 0 records
[]
Partition 8: 0 records
[]
Partition 9: 1 records
[Row(id=4, name='Dave', age=40)]
Partition 10: 0 records
[]
Partition 11: 1 records
[Row(id=5, name='Eve', age=45)]
Partition 12: 0 records
[]
Partition 13: 1 records
[Row(id=6, name='Frank', age=50)]


In [11]:
# Round-robin data partitioning
df_round = df.repartition(5)
# Range data partitioning
df_range = df.repartitionByRange(3,"Name")
# Hash data partitioning
column_hash = "Id"
df_hash = df.repartition(column_hash)

In [12]:
print_partitions(df_round)

####### NUMBER OF PARTITIONS: 5
Partition 0: 1 records
[Row(id=6, name='Frank', age=50)]
Partition 1: 1 records
[Row(id=3, name='Charlie', age=35)]
Partition 2: 0 records
[]
Partition 3: 1 records
[Row(id=5, name='Eve', age=45)]
Partition 4: 3 records
[Row(id=1, name='Alice', age=25), Row(id=2, name='Bob', age=30), Row(id=4, name='Dave', age=40)]


In [13]:
print_partitions(df_range)

####### NUMBER OF PARTITIONS: 3
Partition 0: 2 records
[Row(id=1, name='Alice', age=25), Row(id=2, name='Bob', age=30)]
Partition 1: 2 records
[Row(id=3, name='Charlie', age=35), Row(id=4, name='Dave', age=40)]
Partition 2: 2 records
[Row(id=5, name='Eve', age=45), Row(id=6, name='Frank', age=50)]


In [14]:
print_partitions(df_hash)

####### NUMBER OF PARTITIONS: 1
Partition 0: 6 records
[Row(id=1, name='Alice', age=25), Row(id=2, name='Bob', age=30), Row(id=3, name='Charlie', age=35), Row(id=4, name='Dave', age=40), Row(id=5, name='Eve', age=45), Row(id=6, name='Frank', age=50)]


In [38]:
# Read csv file and load into a dataframe
df = spark.read.csv("bank.csv",header=True)

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#006DAE">TODO: </strong> How many partitions the dataframe have?</div>

In [39]:
df.rdd.getNumPartitions()

1

<a class="anchor" id="lab-task-3"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:##FF5555">3. Lab Task: </strong> Implement Range and Hash Partitioning techniques for the new dataset and display the partitions. COMPLETE THE CODE BELOW.</div>

In [40]:
## We can specify how many partitions or what kind of partitioning we want for a dataframe
# Round-robin data partitioning
df_round = df.repartition(5)
# Range data partitioning
df_range = df.repartitionByRange(3,"age")
# Hash data partitioning
column_hash = "age"
df_hash = df.repartition(column_hash)

In [41]:
## Print the number of partitions for each dataframe
print(f"----- NUMBER OF PARTITIONS df_round: {df_round.rdd.getNumPartitions()}")
print(f"----- NUMBER OF PARTITIONS df_range: {df_range.rdd.getNumPartitions()}")
print(f"----- NUMBER OF PARTITIONS df_hash: {df_hash.rdd.getNumPartitions()}")

## Verifying the number of partitions for the dataframe with hash partitioning it would indicate 200.
## One important thing is that by default, when the number of partitions are not indicated,
## The default number of partitions is 200

## However, most of the partitions for df_hash are empty.

            

----- NUMBER OF PARTITIONS df_round: 5
----- NUMBER OF PARTITIONS df_range: 3
----- NUMBER OF PARTITIONS df_hash: 200


<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#006DAE">TODO: </strong> Complete the above code to show the values for each partition</div>

In [15]:
## You can verify the partitioning and the query plan when an action is performed with the function explain()
# Query plan for df_round
df_round.explain()
# Query plan for df_range
df_range.explain()
# Query plan for df_hash
df_hash.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ShuffleQueryStage 0
   +- Exchange RoundRobinPartitioning(5), REPARTITION_BY_NUM, [plan_id=46]
      +- *(1) Scan ExistingRDD[id#76L,name#77,age#78L]
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(5), REPARTITION_BY_NUM, [plan_id=41]
   +- Scan ExistingRDD[id#76L,name#77,age#78L]


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ShuffleQueryStage 0
   +- Exchange rangepartitioning(Name#77 ASC NULLS FIRST, 3), REPARTITION_BY_NUM, [plan_id=63]
      +- *(1) Scan ExistingRDD[id#76L,name#77,age#78L]
+- == Initial Plan ==
   Exchange rangepartitioning(Name#77 ASC NULLS FIRST, 3), REPARTITION_BY_NUM, [plan_id=57]
   +- Scan ExistingRDD[id#76L,name#77,age#78L]


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   AQEShuffleRead coalesced
   +- ShuffleQueryStage 0
      +- Exchange hashpartitioning(Id#76L, 200), REPARTITION_BY_COL, [plan_id=81]
       

In [ ]:
#写入
def print_partition(partition):
    for record in partition:
        print(record)

df_round.rdd.mapPartitions(print_partition).collect()

df_range.rdd.mapPartitions(print_partition).collect()


df_hash.rdd.mapPartitions(print_partition).collect()

### Parallel Search using Spark Dataframe <a class="anchor" id="parallel_search_df"></a>

We will perform the same filtering criteria as in section 1. This time the logic won't be implemented by us but just declare by using the functions of the Spark Dataframe API to perform the same queries. Thus, we should obtain the same results as before.

Furthermore, now you will need to see in the Spark UI, the RDD DAG Visualisation and the Execution Plan of the queries performed with the function explain() as we did previously.

<a class="anchor" id="lab-task-4"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:##FF5555">4. Lab Task: </strong> Complete the code in the given cell below to implement the given conditions.</div>

In [43]:
df_round.show(3)

+---+----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+-------+
|age|       job|marital|education|default|balance|housing|loan| contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+---+----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+-------+
| 26|   student| single| tertiary|     no|  10086|     no|  no|cellular|  8|  jul|     475|       3|   78|       2| success|     no|
| 32|technician| single|secondary|     no|   -114|     no|  no|cellular| 12|  aug|     155|       2|   -1|       0| unknown|     no|
| 55| housemaid|married|  primary|     no|     10|     no|  no|cellular|  4|  feb|     174|       3|   -1|       0| unknown|     no|
+---+----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+-------+
only showing top 3 rows



In [4]:
from pyspark.sql.functions import col

bank_df = spark.read.csv("bank.csv",header=True)

: 

In [ ]:
# Using the Spark Dataframe API we can obtain the dataframe for a csv file
# We already created dataframes with different types of partitioning
# Choose one of them to work with and perform the queries made in section 1
from pyspark.sql.functions import col

bank_df = spark.read.csv("bank.csv",header=True)

bank_df.show(5)

## The functions to filter in dataframes are similar to the functions in RDD. Thus, the steps are:
# 1. Search the records with balance between 1000 and 2000
bank_df = bank_df.filter(col("balance")>1000)\
            .filter(col("balance")<2000)
# TODO: 
# 2. Also in the same dataframe, search the records with primary or secondary education and age less than 30
bank_df = bank_df.filter(((col("education") == "primary") | (col("education") == "secondary")) & (col("age") < 30))

# TODO:
# 3. Also filter with those who are married
bank_df = bank_df.filter(col("marital") == "married")

# TODO:
# 4. Display the previous attributes plus the information of day, month and deposit, select 'balance','education','age','marital','day','month','deposit'
# bank_df = bank_df.select()
bank_df = bank_df.select("balance", "education", "age", "marital", "day", "month", "deposit")

# 5. Display the records
# 自己选一下展示几行，随意
bank_df.show(5)

+---+----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|age|       job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+---+----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
| 59|    admin.|married|secondary|     no|   2343|    yes|  no|unknown|  5|  may|    1042|       1|   -1|       0| unknown|    yes|
| 56|    admin.|married|secondary|     no|     45|     no|  no|unknown|  5|  may|    1467|       1|   -1|       0| unknown|    yes|
| 41|technician|married|secondary|     no|   1270|    yes|  no|unknown|  5|  may|    1389|       1|   -1|       0| unknown|    yes|
| 55|  services|married|secondary|     no|   2476|    yes|  no|unknown|  5|  may|     579|       1|   -1|       0| unknown|    yes|
| 54|    admin.|married| tertiary|     no|    184|     no|  no|unknown|  5| 

In [45]:
#### Query and partition information
print_partitions(bank_df)
#### Execution Plan for query with multiple filter conditions
bank_df.explain()

####### NUMBER OF PARTITIONS: 5
Partition 0: 3 records
[Row(balance='1303', education='secondary', age='27', marital='married', day='21', month='may', deposit='no'), Row(balance='1596', education='secondary', age='27', marital='married', day='1', month='sep', deposit='yes'), Row(balance='1020', education='secondary', age='28', marital='married', day='28', month='may', deposit='no')]
Partition 1: 4 records
[Row(balance='1222', education='secondary', age='24', marital='married', day='20', month='apr', deposit='no'), Row(balance='1782', education='secondary', age='25', marital='married', day='19', month='jun', deposit='no'), Row(balance='1386', education='secondary', age='29', marital='married', day='28', month='may', deposit='no'), Row(balance='1180', education='secondary', age='29', marital='married', day='17', month='apr', deposit='yes')]
Partition 2: 4 records
[Row(balance='1137', education='secondary', age='28', marital='married', day='6', month='feb', deposit='no'), Row(balance='159

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#006DAE">TODO: </strong> Repeat the same query with different partitioning strategies <strong>(Round-Robin, Range and Hash)</strong> and compare its query execution plan plus its information in the <strong>Spark UI.</strong> </div>

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:##333333">EXAMPLE: </strong>Obtain also the <code>max/min</code> as you did in RDDs but now using <strong>Spark DataFrame</strong>. Does it return the same value as in section 1? Also, check its execution plan and the information in Spark UI.</div>

In [46]:
#### SOLUTION:
bank_max_balance = df_round.selectExpr("int(balance)").selectExpr("max(balance)").collect()
print(bank_max_balance)

[Row(max(balance)=81204)]


### Parallel Search using SQL language in Spark  <a class="anchor" id="parallel_search_sparksql"></a>
#### Spark SQL
Spark SQL is a Spark module for structured data processing. It provides a programming abstraction called DataFrames and can also act as a distributed SQL query engine. It enables unmodified Hadoop Hive queries to run up to 100x faster on existing deployments and data. It also provides powerful integration with the rest of the Spark ecosystem (e.g., integrating SQL query processing with machine learning). <a href="https://databricks.com/glossary/what-is-spark-sql#:~:text=Spark%20SQL%20is%20a%20Spark,on%20existing%20deployments%20and%20data">[Read More]</a>.

A view can be created from a dataframe in order to use SQL queries to search data. In this section, you will use SQL statements to perform search queries in the views that will be registered from the dataframes we created in the previous section.

In [ ]:
df = spark.read.csv("bank.csv",header=True)
df.show(5)

In [18]:
df = spark.read.csv("bank.csv",header=True)
# register the original DataFrame as a temp view so that we can query it using SQL
df.createOrReplaceTempView("df_sql")
filter_sql = spark.sql('''
  SELECT age,education,balance,day,month,deposit
  FROM df_sql
  WHERE balance between 1000 and 2000
  AND education in ('secondary','primary')
  AND age < 30
  AND marital = 'married'
''')
# filter_sql.explain()
filter_sql.collect()

[Row(age='29', education='secondary', balance='1135', day='17', month='feb', deposit='yes'),
 Row(age='27', education='secondary', balance='1293', day='8', month='apr', deposit='yes'),
 Row(age='29', education='secondary', balance='1180', day='17', month='apr', deposit='yes'),
 Row(age='28', education='secondary', balance='1086', day='20', month='apr', deposit='yes'),
 Row(age='26', education='secondary', balance='1595', day='15', month='jun', deposit='yes'),
 Row(age='27', education='secondary', balance='1596', day='1', month='sep', deposit='yes'),
 Row(age='28', education='secondary', balance='1595', day='9', month='sep', deposit='yes'),
 Row(age='27', education='secondary', balance='1595', day='29', month='dec', deposit='yes'),
 Row(age='26', education='secondary', balance='1417', day='6', month='jun', deposit='no'),
 Row(age='23', education='secondary', balance='1309', day='3', month='jun', deposit='no'),
 Row(age='24', education='secondary', balance='1222', day='20', month='apr', 

<a class="anchor" id="lab-task-6"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#333333">EXAMPLE: </strong>Obtain also the <code>max/min</code> as you did in RDDs and DataFrames, but now using <strong>Spark SQL</strong>. Does it return the same value as in previous cases? Also, check its execution plan and the information in Spark UI.</div>

In [48]:
#### SOLUTION:
max_sql = spark.sql('''
  SELECT MAX(CAST(balance AS INT)) as max_balance
  FROM df_sql
''')
# Check the result obtained
max_sql.collect()

[Row(max_balance=81204)]

In [19]:
sc.stop()

**Congratulations on finishing this activity!**

Having practiced today's activities, we're now ready to embark on a trip of the rest of exiciting FIT5202 activities! See you next week!